In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
import seaborn as sns

import folium
from folium import Choropleth
from folium.features import GeoJson, GeoJsonTooltip

import branca
import branca.colormap as cm

import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
aq = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Processed data/air_quality_processed_with_annual_2024_2025.csv')

In [3]:
print("Columns in 'aq' dataframe:", aq.columns)

Columns in 'aq' dataframe: Index(['Unnamed: 0', 'Geo Join ID', 'Geo Place Name', 'Geo Type Name',
       'Start_Date', 'Time Period', 'Name', 'Measure Info', 'Data Value'],
      dtype='object')


In [4]:
aq = aq.drop(columns=['Unnamed: 0'], errors='ignore')

In [ ]:
aq = aq.rename(columns={
    "Geo Join ID": "geo_join_id",
    "Geo Place Name": "geo_place_name",
    "Geo Type Name": "geo_type_name",
    "Start_Date": "start_date",
    "Time Period": "time_period",
    "Name": "pollutant_name",  # Rename 'Name' to 'pollutant_name'
    "Measure Info": "measure_info",
    "Data Value": "data_value"
})

In [ ]:
print("Columns in 'aq' dataframe after cleaning:", aq.columns)

In [ ]:
# Ensure start_date is a proper datetime
aq["start_date"] = pd.to_datetime(aq["start_date"], errors="coerce")

# Extract year and month (handy later for trends / merging)
aq["year"] = aq["start_date"].dt.year
aq["month"] = aq["start_date"].dt.month

# Quick check again
aq[["start_date", "year", "month"]].head()

In [ ]:
aq_boro = aq[aq["geo_type_name"] == "Borough"].copy()

In [ ]:
boro_map = {
    "Bronx": "BRONX",
    "Brooklyn": "BROOKLYN",
    "Manhattan": "MANHATTAN",
    "Queens": "QUEENS",
    "Staten Island": "STATEN ISLAND"
}
aq_boro["BORO_NM_IMPUTED"] = aq_boro["geo_place_name"].map(boro_map)
aq_boro["BORO_NM_IMPUTED"] = aq_boro["BORO_NM_IMPUTED"].fillna("UNKNOWN")

In [ ]:
print("Columns in 'aq_boro' dataframe:", aq_boro.columns)

In [ ]:
# Ensure 'year_from_period' is an integer
aq_boro_key['year_from_period'] = aq_boro_key['year_from_period'].astype(int)

In [6]:
# Group by season and borough
season_avg = (
    aq.groupby(["season", "BORO_NM_IMPUTED"], as_index=False)["data_value"]
      .mean()
)

# Define a clean season order if applicable
season_order = ["Winter", "Spring", "Summer", "Fall", "Annual"]
season_order = [s for s in season_order if s in season_avg["season"].unique()]

#O#O#O#O#O#O#O#O

# Extract only YEAR from time_period
aq_boro["year_from_period"] = aq_boro["time_period"].str.extract(r'(\d{4})').astype(float)

pollutant_to_plot = "Fine particles (PM 2.5)"
df = aq_boro[aq_boro["pollutant_name"] == pollutant_to_plot].copy()

yearly_avg = (
    df.groupby(["year_from_period", "BORO_NM_IMPUTED"], as_index=False)["data_value"]
      .mean()
)

#O#O#O#O#O#O#O#O#O


# =========================================================
# Choose pollutant to analyze
# =========================================================
pollutant_to_plot = "Fine particles (PM 2.5)"   

# Filter once, reuse for both plots
aq_poll = aq_boro[aq_boro["pollutant_name"] == pollutant_to_plot].copy()

# =========================================================
# 1) SEASONAL PATTERN BY BOROUGH (BAR PLOT)
# =========================================================

# Extract season text from time_period (e.g. "Winter", "Summer", "Annual")
aq_poll["season"] = aq_poll["time_period"].str.extract(r'([A-Za-z]+)', expand=False)

# Group by season and borough
season_avg = (
    aq_poll
    .groupby(["season", "BORO_NM_IMPUTED"], as_index=False)["data_value"]
    .mean()
)

# Clean season order based on what actually exists
season_order = ["Winter", "Spring", "Summer", "Fall", "Annual"]
season_order = [s for s in season_order if s in season_avg["season"].unique()]


# Extract year from time_period (e.g. "2009 Annual" -> 2009)
aq_poll["year_from_period"] = (
    aq_poll["time_period"]
    .str.extract(r'(\d{4})', expand=False)
    .astype(float)
)

# Drop rows without a valid year
yearly = aq_poll.dropna(subset=["year_from_period"]).copy()

# Group by year and borough
yearly_avg = (
    yearly
    .groupby(["year_from_period", "BORO_NM_IMPUTED"], as_index=False)["data_value"]
    .mean()
)

#O#O#O#OO#O#O#O#O

# ------------------------------------------------------------
# 1. Make sure we have a year column
# ------------------------------------------------------------
if "year_from_period" not in aq_boro.columns:
    aq_boro["year_from_period"] = (
        aq_boro["time_period"]
        .str.extract(r'(\d{4})', expand=False)
        .astype(float)
    )

# ------------------------------------------------------------
# 2. Dropdown for pollutant selection
# ------------------------------------------------------------
pollutants = sorted(aq_boro["pollutant_name"].unique())

pollutant_dropdown = widgets.Dropdown(
    options=pollutants,
    description="Pollutant:",
    value=pollutants[0],
    layout=widgets.Layout(width="60%")
)

# (Optional) dropdown to filter a single borough or show all
boroughs = ["ALL"] + sorted(aq_boro["BORO_NM_IMPUTED"].unique())
borough_dropdown = widgets.Dropdown(
    options=boroughs,
    description="Borough:",
    value="ALL",
    layout=widgets.Layout(width="40%")
)

# ------------------------------------------------------------
# 3. Plotting function (called whenever dropdown changes)
# ------------------------------------------------------------
def plot_yearly_trend(pollutant_to_plot, borough_choice):
    df = aq_boro[aq_boro["pollutant_name"] == pollutant_to_plot].copy()
    df = df.dropna(subset=["year_from_period"])

    if borough_choice != "ALL":
        df = df[df["BORO_NM_IMPUTED"] == borough_choice]

    if df.empty:
        print("No data available for this selection.")
        return

    yearly_avg = (
        df.groupby(["year_from_period", "BORO_NM_IMPUTED"], as_index=False)["data_value"]
          .mean()
    )

#O#O#O#O#O#O#O#O#O#O#

# =========================================================
# 2. Clean columns, parse dates
# =========================================================
aq = aq.rename(columns={
    "Geo Join ID": "geo_join_id",
    "Geo Place Name": "geo_place_name",
    "Geo Type Name": "geo_type_name",
    "Start_Date": "start_date",
    "Time Period": "time_period",
    "Name": "pollutant_name",
    "Measure Info": "measure_info",
    "Data Value": "data_value"
})

aq["start_date"] = pd.to_datetime(aq["start_date"], errors="coerce")

# Extract year from the time_period text (e.g. "2009 Annual")
aq["year_from_period"] = (
    aq["time_period"]
      .str.extract(r'(\d{4})', expand=False)
      .astype(float)
)

# =========================================================
# 3. Keep only Borough-level data & standardize borough names
# =========================================================
aq_boro = aq[aq["geo_type_name"] == "Borough"].copy()

boro_map = {
    "Bronx": "BRONX",
    "Brooklyn": "BROOKLYN",
    "Manhattan": "MANHATTAN",
    "Queens": "QUEENS",
    "Staten Island": "STATEN ISLAND"
}
aq_boro["BORO_NM_IMPUTED"] = aq_boro["geo_place_name"].map(boro_map)
aq_boro["BORO_NM_IMPUTED"] = aq_boro["BORO_NM_IMPUTED"].fillna("UNKNOWN")

print("Borough-level rows:", aq_boro.shape[0])
print("Pollutants in borough data:\n", aq_boro["pollutant_name"].unique())

# =========================================================
# 4. Define the key pollutants we care about
# =========================================================
important_pollutants = [
    "Fine particles (PM 2.5)",
    "Nitrogen dioxide (NO2)",
    "Ozone (O3)"
]

# Filter aq_boro to keep only these (optional but cleaner)
aq_boro_key = aq_boro[aq_boro["pollutant_name"].isin(important_pollutants)].copy()

# =========================================================
# 5. Build dropdown widgets (pollutant + borough)
# =========================================================
pollutant_dropdown = widgets.Dropdown(
    options=important_pollutants,
    description="Pollutant:",
    value=important_pollutants[0],
    layout=widgets.Layout(width="60%")
)

borough_options = ["ALL"] + sorted(aq_boro_key["BORO_NM_IMPUTED"].unique())
borough_dropdown = widgets.Dropdown(
    options=borough_options,
    description="Borough:",
    value="ALL",
    layout=widgets.Layout(width="40%")
)

# =========================================================
# 6. Plot function: yearly trend line plot
# =========================================================
def plot_yearly_trend(pollutant_to_plot, borough_choice):
    # filter by pollutant
    df = aq_boro_key[aq_boro_key["pollutant_name"] == pollutant_to_plot].copy()
    df = df.dropna(subset=["year_from_period"])

    # optional borough filter
    if borough_choice != "ALL":
        df = df[df["BORO_NM_IMPUTED"] == borough_choice]

    if df.empty:
        print("No data available for this selection.")
        return

    # yearly mean per borough
    yearly_avg = (
        df.groupby(["year_from_period", "BORO_NM_IMPUTED"], as_index=False)["data_value"]
          .mean()
    )


KeyError: 'season'

In [ ]:
tier1_pollutants = [
    "Fine particles (PM 2.5)",
    "Nitrogen dioxide (NO2)",
    "Ozone (O3)"
]

# Keep only Tier-1 pollutants for the base data
aq_boro_key = aq_boro[aq_boro["pollutant_name"].isin(tier1_pollutants)].copy()

# ---- Create composite "index" across all 3 Tier-1 pollutants ----
# Average data_value over the three pollutants for each borough-year
index_label = "Key pollutants index (PM2.5 + NO2 + O3 avg)"

index_df = (
    aq_boro_key
    .dropna(subset=["year_from_period"])
    .groupby(["BORO_NM_IMPUTED", "year_from_period"], as_index=False)
    .agg({"data_value": "mean"})
)

# Add the 'virtual' pollutant name
index_df["pollutant_name"] = index_label

# (Optional) attach a dummy time_period so structure matches
index_df["time_period"] = index_df["year_from_period"].astype(int).astype(str) + " (index)"

# Concatenate: Tier-1 pollutants + composite index
aq_boro_key = pd.concat([aq_boro_key, index_df], ignore_index=True)

# Final list of pollutant options for dropdown
important_pollutants = tier1_pollutants + [index_label]

print("Important pollutants/options:")
for p in important_pollutants:
    print("-", p)

# =========================================================
# 5. Build dropdown widgets (pollutant + borough)
# =========================================================
pollutant_dropdown = widgets.Dropdown(
    options=important_pollutants,
    description="Pollutant:",
    value=important_pollutants[0],
    layout=widgets.Layout(width="65%")
)

borough_options = ["ALL"] + sorted(aq_boro_key["BORO_NM_IMPUTED"].unique())
borough_dropdown = widgets.Dropdown(
    options=borough_options,
    description="Borough:",
    value="ALL",
    layout=widgets.Layout(width="35%")
)

# =========================================================
# 6. Plot function: yearly trend line plot
# =========================================================
def plot_yearly_trend(pollutant_to_plot, borough_choice):
    # filter by pollutant (includes the composite index now)
    df = aq_boro_key[aq_boro_key["pollutant_name"] == pollutant_to_plot].copy()
    df = df.dropna(subset=["year_from_period"])

    # optional borough filter
    if borough_choice != "ALL":
        df = df[df["BORO_NM_IMPUTED"] == borough_choice]

    if df.empty:
        print("No data available for this selection.")
        return

    # yearly mean per borough
    yearly_avg = (
        df.groupby(["year_from_period", "BORO_NM_IMPUTED"], as_index=False)["data_value"]
          .mean()
    )

    plt.figure(figsize=(12, 6))
    sns.lineplot(
        data=yearly_avg,
        x="year_from_period",
        y="data_value",
        hue="BORO_NM_IMPUTED",
        marker="o"
    )

    borough_label = "" if borough_choice == "ALL" else f" – {borough_choice}"
    plt.title(f"Yearly Trend of {pollutant_to_plot} by Borough{borough_label}")
    plt.xlabel("Year")
    plt.ylabel("Average Value")
    plt.tight_layout()
    plt.show()

# =========================================================
# 7. Connect widgets to the plot function & display UI
# =========================================================
ui = widgets.HBox([pollutant_dropdown, borough_dropdown])
out = widgets.interactive_output(
    plot_yearly_trend,
    {"pollutant_to_plot": pollutant_dropdown,
     "borough_choice": borough_dropdown}
)

display(ui, out)